In [2]:
# ============================================================
# V3 — SELF-CONTAINED
# Rebuild V2 + Cost/Latency/Quality-Aware Routing
# ============================================================

import json
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sentence_transformers import SentenceTransformer

from scipy.sparse import csr_matrix, hstack


# ============================================================
# 1. PATHS
# ============================================================

BASE = Path("/home/sagemaker-user/adaptive-llm-routing")

DATA_PATH = (
    BASE / "notebooks/data/evaluation/"
    "v1_complexity_labels.json"
)

V3_DIR = (
    BASE / "notebooks/data/evaluation/v3"
)

V3_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 2. LOAD ORIGINAL V1 DATA
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

texts = []
labels = []

for item in data:

    text = item.get("instruction", "").strip()
    label = item.get("complexity", "").strip().lower()

    if text and label in {"simple", "medium", "complex"}:
        texts.append(text)
        labels.append(label)

print("=" * 80)
print("V3 — SELF-CONTAINED ROUTER")
print("=" * 80)

print(f"\nDataset: {len(texts)}")
print("Labels:", Counter(labels))


# ============================================================
# 3. SAME V2 TRAIN/VALIDATION SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print(f"\nTraining:   {len(X_train)}")
print(f"Validation: {len(X_test)}")


# ============================================================
# 4. REBUILD V2 TF-IDF
# ============================================================

print("\nLoading TF-IDF...")

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=10000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)


# ============================================================
# 5. REBUILD V2 MINILM
# ============================================================

print("\nLoading MiniLM...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Encoding training data...")

X_train_emb = embedding_model.encode(
    X_train,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ============================================================
# 6. QUERY FEATURES
# ============================================================

def extract_query_features(texts):

    features = []

    for text in texts:

        words = text.split()

        word_count = len(words)

        char_count = len(text)

        sentence_count = max(
            1,
            sum(
                text.count(x)
                for x in [".", "?", "!"]
            )
        )

        question_mark = int("?" in text)

        newline_count = text.count("\n")

        avg_word_length = (
            np.mean([len(w) for w in words])
            if words else 0
        )

        features.append([
            word_count,
            char_count,
            sentence_count,
            question_mark,
            newline_count,
            avg_word_length
        ])

    return np.asarray(
        features,
        dtype=np.float32
    )


X_train_stats = extract_query_features(
    X_train
)

scaler = StandardScaler()

X_train_stats = scaler.fit_transform(
    X_train_stats
)


# ============================================================
# 7. COMBINE V2 FEATURES
# ============================================================

X_train_combined = hstack([
    X_train_tfidf,
    csr_matrix(X_train_emb),
    csr_matrix(X_train_stats)
])

print(
    f"\nCombined V2 features: "
    f"{X_train_combined.shape}"
)


# ============================================================
# 8. TRAIN V2 CLASSIFIER
# ============================================================

print("\nTraining V2 classifier...")

model = LogisticRegression(
    max_iter=1500,
    class_weight="balanced",
    random_state=42
)

model.fit(
    X_train_combined,
    y_train
)

print("V2 model ready.")


# ============================================================
# 9. V3 MODEL PROFILES
# ============================================================
# SIMULATED values for experimentation.
# These will later be replaced by real measurements.

model_profiles = {

    "small_model": {
        "quality": 0.82,
        "latency_ms": 200,
        "cost": 0.10
    },

    "medium_model": {
        "quality": 0.91,
        "latency_ms": 500,
        "cost": 0.30
    },

    "large_model": {
        "quality": 0.97,
        "latency_ms": 1200,
        "cost": 1.00
    }
}


# ============================================================
# 10. NORMALIZATION
# ============================================================

costs = np.array([
    p["cost"]
    for p in model_profiles.values()
])

latencies = np.array([
    p["latency_ms"]
    for p in model_profiles.values()
])

qualities = np.array([
    p["quality"]
    for p in model_profiles.values()
])

min_cost = costs.min()
max_cost = costs.max()

min_latency = latencies.min()
max_latency = latencies.max()

min_quality = qualities.min()
max_quality = qualities.max()


def normalize(value, minimum, maximum):

    if maximum == minimum:
        return 0.0

    return (
        value - minimum
    ) / (
        maximum - minimum
    )


# ============================================================
# 11. V3 SCORING
# ============================================================

def v3_score_model(
    model_name,
    complexity,
    confidence,
    quality_weight=0.50,
    cost_weight=0.25,
    latency_weight=0.25
):

    profile = model_profiles[model_name]

    quality = normalize(
        profile["quality"],
        min_quality,
        max_quality
    )

    cost = normalize(
        profile["cost"],
        min_cost,
        max_cost
    )

    latency = normalize(
        profile["latency_ms"],
        min_latency,
        max_latency
    )

    complexity_requirement = {
        "simple": 0.30,
        "medium": 0.65,
        "complex": 0.95
    }[complexity]

    quality_gap = max(
        0,
        complexity_requirement -
        profile["quality"]
    )

    quality_penalty = (
        quality_gap * 2.0
    )

    uncertainty_penalty = (
        (1 - confidence) * 0.10
    )

    score = (
        quality_weight * quality
        - cost_weight * cost
        - latency_weight * latency
        - quality_penalty
        - uncertainty_penalty
    )

    return score


# ============================================================
# 12. V3 ROUTER
# ============================================================

def route_v3(query):

    # V2 TF-IDF
    tfidf_features = tfidf.transform(
        [query]
    )

    # V2 embedding
    embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # V2 query features
    stats = extract_query_features(
        [query]
    )

    stats = scaler.transform(
        stats
    )

    # Combine
    features = hstack([
        tfidf_features,
        csr_matrix(embedding),
        csr_matrix(stats)
    ])

    # V2 prediction
    complexity = model.predict(
        features
    )[0]

    probabilities = model.predict_proba(
        features
    )[0]

    confidence = float(
        np.max(probabilities)
    )

    # V3 score every model
    scores = {}

    for model_name in model_profiles:

        scores[model_name] = (
            v3_score_model(
                model_name,
                complexity,
                confidence
            )
        )

    selected_model = max(
        scores,
        key=scores.get
    )

    return {
        "query": query,
        "complexity": complexity,
        "confidence": confidence,
        "selected_model": selected_model,
        "scores": scores
    }


# ============================================================
# 13. TEST QUERIES
# ============================================================

test_queries = [

    "What is Python?",

    "What is the capital of France?",

    "What is 25 multiplied by 4?",

    "Compare Python and Java for backend development.",

    "Explain the advantages and disadvantages of remote work.",

    "What are the main differences between supervised "
    "and unsupervised learning?",

    "Design a fraud detection system for a bank that "
    "minimizes false positives while detecting previously "
    "unseen fraud patterns.",

    "Design an AI system that processes millions of "
    "financial transactions in real time, detects fraud, "
    "explains its decisions, and adapts to changing "
    "fraud patterns.",

    "Analyze the tradeoffs between cost, latency, accuracy, "
    "and scalability when choosing between several large "
    "language models for a production application."
]


# ============================================================
# 14. RUN V3
# ============================================================

results = []

for query in test_queries:

    result = route_v3(query)

    results.append({
        "query": query,
        "complexity": result["complexity"],
        "confidence": round(
            result["confidence"],
            4
        ),
        "selected_model": result[
            "selected_model"
        ],
        "small_score": round(
            result["scores"]["small_model"],
            4
        ),
        "medium_score": round(
            result["scores"]["medium_model"],
            4
        ),
        "large_score": round(
            result["scores"]["large_model"],
            4
        )
    })


results_df = pd.DataFrame(
    results
)


# ============================================================
# 15. DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("V3 ROUTING RESULTS")
print("=" * 100)

display(results_df)


print("\n" + "=" * 80)
print("MODEL PROFILES")
print("=" * 80)

display(
    pd.DataFrame(
        model_profiles
    ).T
)


# ============================================================
# 16. SAVE V3 ARTIFACTS
# ============================================================

with open(
    V3_DIR / "model_profiles.json",
    "w"
) as f:

    json.dump(
        model_profiles,
        f,
        indent=2
    )

results_df.to_csv(
    V3_DIR / "v3_routing_test.csv",
    index=False
)

print("\nV3 artifacts saved to:")
print(V3_DIR)

/home/sagemaker-user/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


V3 — SELF-CONTAINED ROUTER

Dataset: 639
Labels: Counter({'simple': 213, 'medium': 213, 'complex': 213})

Training:   511
Validation: 128

Loading TF-IDF...

Loading MiniLM...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1283.70it/s]


Encoding training data...


Batches: 100%|██████████| 16/16 [00:04<00:00,  3.25it/s]



Combined V2 features: (511, 916)

Training V2 classifier...
V2 model ready.

V3 ROUTING RESULTS


,query,complexity,confidence,selected_model,small_score,medium_score,large_score
0,What is Python?,simple,0.5078,medium_model,-0.0492,0.1202,-0.0492
1,What is the capital of France?,complex,0.6696,medium_model,-0.2930,0.0564,-0.0330
2,What is 25 multiplied by 4?,complex,0.4884,medium_model,-0.3112,0.0383,-0.0512
3,Compare Python and Java for backend development.,simple,0.4874,medium_model,-0.0513,0.1182,-0.0513
4,Explain the advantages and disadvantages of re...,medium,0.4010,medium_model,-0.0599,0.1095,-0.0599
5,What are the main differences between supervis...,complex,0.4044,medium_model,-0.3196,0.0299,-0.0596
6,Design a fraud detection system for a bank tha...,complex,0.5254,medium_model,-0.3075,0.0420,-0.0475
7,Design an AI system that processes millions of...,medium,0.4395,medium_model,-0.0561,0.1134,-0.0561
8,"Analyze the tradeoffs between cost, latency, a...",complex,0.5023,medium_model,-0.3098,0.0397,-0.0498



MODEL PROFILES


,quality,latency_ms,cost
small_model,0.82,200.0,0.1
medium_model,0.91,500.0,0.3
large_model,0.97,1200.0,1.0



V3 artifacts saved to:
/home/sagemaker-user/adaptive-llm-routing/notebooks/data/evaluation/v3


In [4]:
# ============================================================
# V3 — FIXED QUALITY / COST / LATENCY ROUTER
# ============================================================

import os
import json
import numpy as np
import pandas as pd


print("=" * 90)
print("V3 — QUALITY / COST / LATENCY AWARE ROUTER")
print("=" * 90)


# ============================================================
# 1. MODEL PROFILES
# ============================================================

model_profiles = {

    "small_model": {
        "quality": 0.82,
        "latency_ms": 200,
        "cost": 0.10
    },

    "medium_model": {
        "quality": 0.91,
        "latency_ms": 500,
        "cost": 0.30
    },

    "large_model": {
        "quality": 0.97,
        "latency_ms": 1200,
        "cost": 1.00
    }
}


# ============================================================
# 2. MINIMUM QUALITY REQUIREMENTS
# ============================================================

quality_requirement = {

    "simple": 0.80,

    "medium": 0.90,

    "complex": 0.95
}


# ============================================================
# 3. COST / LATENCY WEIGHTS
# ============================================================

COST_WEIGHT = 0.55
LATENCY_WEIGHT = 0.45


# ============================================================
# 4. NORMALIZE COST AND LATENCY
# ============================================================

cost_values = np.array([
    profile["cost"]
    for profile in model_profiles.values()
])

latency_values = np.array([
    profile["latency_ms"]
    for profile in model_profiles.values()
])


def min_max_normalize(value, minimum, maximum):

    if maximum == minimum:
        return 0.0

    return (value - minimum) / (maximum - minimum)


for model_name, profile in model_profiles.items():

    profile["normalized_cost"] = min_max_normalize(
        profile["cost"],
        cost_values.min(),
        cost_values.max()
    )

    profile["normalized_latency"] = min_max_normalize(
        profile["latency_ms"],
        latency_values.min(),
        latency_values.max()
    )


# ============================================================
# 5. V3 ROUTER
# ============================================================

def route_v3(complexity, confidence):

    required_quality = quality_requirement[complexity]

    # --------------------------------------------------------
    # Slight confidence-aware safety margin
    #
    # Low confidence -> slightly more conservative routing.
    # --------------------------------------------------------

    confidence_margin = (
        (1.0 - confidence) * 0.02
    )

    adjusted_requirement = min(
        1.0,
        required_quality + confidence_margin
    )


    # --------------------------------------------------------
    # STEP 1:
    # Find models that actually satisfy quality requirement.
    # --------------------------------------------------------

    eligible_models = {}

    for model_name, profile in model_profiles.items():

        if profile["quality"] >= adjusted_requirement:

            eligible_models[model_name] = profile


    # --------------------------------------------------------
    # STEP 2:
    # Safety fallback.
    #
    # If no model qualifies, use highest-quality model.
    # --------------------------------------------------------

    if not eligible_models:

        best_model = max(
            model_profiles.items(),
            key=lambda item: item[1]["quality"]
        )[0]

        return best_model, {
            "required_quality": adjusted_requirement,
            "eligible_models": [],
            "scores": {},
            "fallback": True
        }


    # --------------------------------------------------------
    # STEP 3:
    # Optimize cost + latency ONLY among eligible models.
    # --------------------------------------------------------

    scores = {}

    for model_name, profile in eligible_models.items():

        operational_score = (

            COST_WEIGHT *
            profile["normalized_cost"]

            +

            LATENCY_WEIGHT *
            profile["normalized_latency"]
        )

        scores[model_name] = operational_score


    # Lower score = better.
    selected_model = min(
        scores,
        key=scores.get
    )


    return selected_model, {
        "required_quality": adjusted_requirement,
        "eligible_models": list(eligible_models.keys()),
        "scores": scores,
        "fallback": False
    }


# ============================================================
# 6. TEST QUERIES
# ============================================================

test_queries = [

    "What is Python?",

    "What is the capital of France?",

    "What is 25 multiplied by 4?",

    "Compare Python and Java for backend development.",

    "Explain the advantages and disadvantages of remote work.",

    "What are the main differences between supervised and unsupervised learning?",

    "Design a fraud detection system for a bank that processes millions of transactions.",

    "Design an AI system that processes millions of financial transactions in real time.",

    "Analyze the tradeoffs between cost, latency, and accuracy in an AI system."
]


# ============================================================
# 7. RUN V2 + V3
# ============================================================

results = []


for query in test_queries:

    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------

    tfidf_features = tfidf.transform(
        [query]
    ).toarray()


    # --------------------------------------------------------
    # MiniLM
    # --------------------------------------------------------

    embeddings = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )


    # --------------------------------------------------------
    # Query statistics
    # --------------------------------------------------------

    query_features = extract_query_features(
        [query]
    )

    query_features_scaled = scaler.transform(
        query_features
    )


    # --------------------------------------------------------
    # Combine V2 features
    # --------------------------------------------------------

    combined_features = np.hstack([

        tfidf_features,

        embeddings,

        query_features_scaled
    ])


    # --------------------------------------------------------
    # V2 complexity prediction
    # --------------------------------------------------------

    complexity = model.predict(
        combined_features
    )[0]


    probabilities = model.predict_proba(
        combined_features
    )[0]


    confidence = float(
        np.max(probabilities)
    )


    # --------------------------------------------------------
    # V3 routing
    # --------------------------------------------------------

    selected_model, routing_info = route_v3(
        complexity,
        confidence
    )


    # --------------------------------------------------------
    # Scores
    # --------------------------------------------------------

    small_score = routing_info["scores"].get(
        "small_model",
        np.nan
    )

    medium_score = routing_info["scores"].get(
        "medium_model",
        np.nan
    )

    large_score = routing_info["scores"].get(
        "large_model",
        np.nan
    )


    # --------------------------------------------------------
    # Save result
    # --------------------------------------------------------

    results.append({

        "query": query,

        "complexity": complexity,

        "confidence": round(
            confidence,
            4
        ),

        "required_quality": round(
            routing_info["required_quality"],
            4
        ),

        "eligible_models":
            ", ".join(
                routing_info["eligible_models"]
            ),

        "selected_model":
            selected_model,

        "small_score":
            round(small_score, 4)
            if not np.isnan(small_score)
            else None,

        "medium_score":
            round(medium_score, 4)
            if not np.isnan(medium_score)
            else None,

        "large_score":
            round(large_score, 4)
            if not np.isnan(large_score)
            else None,

        "fallback":
            routing_info["fallback"]
    })


# ============================================================
# 8. RESULTS
# ============================================================

results_df = pd.DataFrame(
    results
)


print("\n")
print("=" * 120)
print("V3 ROUTING RESULTS")
print("=" * 120)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 9. MODEL SELECTION DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("MODEL SELECTION DISTRIBUTION")
print("=" * 80)

print(
    results_df[
        "selected_model"
    ].value_counts()
)


# ============================================================
# 10. ELIGIBILITY DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("ELIGIBLE MODEL DISTRIBUTION")
print("=" * 80)

for _, row in results_df.iterrows():

    print(
        f"{row['complexity']:8s} | "
        f"confidence={row['confidence']:.4f} | "
        f"required={row['required_quality']:.4f} | "
        f"eligible={row['eligible_models']} | "
        f"selected={row['selected_model']}"
    )


# ============================================================
# 11. MODEL PROFILES
# ============================================================

print("\n")
print("=" * 80)
print("MODEL PROFILES")
print("=" * 80)

profile_df = pd.DataFrame(
    model_profiles
).T

print(
    profile_df[
        [
            "quality",
            "latency_ms",
            "cost",
            "normalized_cost",
            "normalized_latency"
        ]
    ]
)


# ============================================================
# 12. SAVE RESULTS
# ============================================================

save_dir = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing/"
    "notebooks/data/evaluation/v3"
)

os.makedirs(
    save_dir,
    exist_ok=True
)


# Save profiles

with open(
    os.path.join(
        save_dir,
        "model_profiles.json"
    ),
    "w"
) as f:

    json.dump(
        model_profiles,
        f,
        indent=2
    )


# Save routing results

results_path = os.path.join(
    save_dir,
    "v3_routing_test.csv"
)

results_df.to_csv(
    results_path,
    index=False
)


print("\n")
print("=" * 80)
print("V3 COMPLETE")
print("=" * 80)

print(
    "Model profiles:",
    os.path.join(
        save_dir,
        "model_profiles.json"
    )
)

print(
    "Routing results:",
    results_path
)

V3 — QUALITY / COST / LATENCY AWARE ROUTER


V3 ROUTING RESULTS
                                                                              query complexity  confidence  required_quality                        eligible_models selected_model  small_score  medium_score  large_score  fallback
                                                                    What is Python?     simple      0.5078            0.8098 small_model, medium_model, large_model    small_model          0.0        0.2572          1.0     False
                                                     What is the capital of France?    complex      0.6696            0.9566                            large_model    large_model          NaN           NaN          1.0     False
                                                        What is 25 multiplied by 4?    complex      0.4884            0.9602                            large_model    large_model          NaN           NaN          1.0     False
                    

In [5]:
# ============================================================
# V3 — FINAL QUALITY / COST / LATENCY AWARE ROUTER
# ============================================================
#
# V2:
#   Query -> complexity + confidence
#
# V3:
#   complexity
#       ↓
#   minimum quality requirement
#       ↓
#   eliminate models below requirement
#       ↓
#   optimize cost + latency among eligible models
#       ↓
#   selected model
#
# NOTE:
# Model quality/cost/latency values are simulated for V3.
# They are NOT measured real-world LLM benchmarks.
# ============================================================


import os
import json
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

print("=" * 90)
print("V3 — FINAL QUALITY / COST / LATENCY AWARE ROUTER")
print("=" * 90)


# ------------------------------------------------------------
# Model profiles
# ------------------------------------------------------------
# These are simulated profiles for the V3 experiment.
#
# quality:
#   expected response quality, 0 → 1
#
# latency_ms:
#   expected response latency
#
# cost:
#   normalized relative cost
#
# Later versions will replace these with measured telemetry.
# ------------------------------------------------------------

model_profiles = {

    "small_model": {
        "quality": 0.82,
        "latency_ms": 200,
        "cost": 0.10
    },

    "medium_model": {
        "quality": 0.91,
        "latency_ms": 500,
        "cost": 0.30
    },

    "large_model": {
        "quality": 0.97,
        "latency_ms": 1200,
        "cost": 1.00
    }
}


# ------------------------------------------------------------
# Minimum acceptable quality by predicted complexity
# ------------------------------------------------------------

quality_requirement = {

    "simple": 0.80,

    "medium": 0.90,

    "complex": 0.95
}


# ------------------------------------------------------------
# Cost / latency optimization weights
# ------------------------------------------------------------
#
# Lower operational score = better.
#
# Cost receives slightly more weight than latency.
# ------------------------------------------------------------

COST_WEIGHT = 0.55
LATENCY_WEIGHT = 0.45


# ============================================================
# 2. NORMALIZE COST AND LATENCY
# ============================================================

cost_values = np.array([
    profile["cost"]
    for profile in model_profiles.values()
])

latency_values = np.array([
    profile["latency_ms"]
    for profile in model_profiles.values()
])


def min_max_normalize(value, minimum, maximum):

    if maximum == minimum:
        return 0.0

    return (
        (value - minimum)
        /
        (maximum - minimum)
    )


for model_name, profile in model_profiles.items():

    profile["normalized_cost"] = min_max_normalize(
        profile["cost"],
        cost_values.min(),
        cost_values.max()
    )

    profile["normalized_latency"] = min_max_normalize(
        profile["latency_ms"],
        latency_values.min(),
        latency_values.max()
    )


# ============================================================
# 3. V3 ROUTING FUNCTION
# ============================================================

def route_v3(complexity):

    # --------------------------------------------------------
    # Determine minimum quality required.
    # --------------------------------------------------------

    if complexity not in quality_requirement:

        raise ValueError(
            f"Unknown complexity level: {complexity}"
        )

    required_quality = quality_requirement[
        complexity
    ]


    # --------------------------------------------------------
    # STEP 1:
    # Filter models using the quality constraint.
    # --------------------------------------------------------

    eligible_models = {}

    for model_name, profile in model_profiles.items():

        if profile["quality"] >= required_quality:

            eligible_models[model_name] = profile


    # --------------------------------------------------------
    # STEP 2:
    # Safety fallback.
    #
    # If no model satisfies the quality requirement,
    # use the highest-quality model available.
    # --------------------------------------------------------

    if not eligible_models:

        selected_model = max(
            model_profiles.items(),
            key=lambda item: item[1]["quality"]
        )[0]

        return selected_model, {
            "required_quality": required_quality,
            "eligible_models": [],
            "scores": {},
            "fallback": True
        }


    # --------------------------------------------------------
    # STEP 3:
    # Optimize cost + latency among eligible models.
    # --------------------------------------------------------

    scores = {}

    for model_name, profile in eligible_models.items():

        operational_score = (

            COST_WEIGHT
            *
            profile["normalized_cost"]

            +

            LATENCY_WEIGHT
            *
            profile["normalized_latency"]
        )

        scores[model_name] = operational_score


    # --------------------------------------------------------
    # STEP 4:
    # Lowest operational score wins.
    # --------------------------------------------------------

    selected_model = min(
        scores,
        key=scores.get
    )


    return selected_model, {
        "required_quality": required_quality,
        "eligible_models": list(
            eligible_models.keys()
        ),
        "scores": scores,
        "fallback": False
    }


# ============================================================
# 4. TEST QUERIES
# ============================================================

test_queries = [

    "What is Python?",

    "What is the capital of France?",

    "What is 25 multiplied by 4?",

    "Compare Python and Java for backend development.",

    "Explain the advantages and disadvantages of remote work.",

    "What are the main differences between supervised and unsupervised learning?",

    "Design a fraud detection system for a bank that processes millions of transactions.",

    "Design an AI system that processes millions of financial transactions in real time.",

    "Analyze the tradeoffs between cost, latency, and accuracy in an AI system."
]


# ============================================================
# 5. RUN V2 PREDICTION + V3 ROUTING
# ============================================================

results = []


for query in test_queries:

    # --------------------------------------------------------
    # V2 FEATURE EXTRACTION
    # --------------------------------------------------------

    tfidf_features = tfidf.transform(
        [query]
    ).toarray()


    embeddings = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        show_progress_bar=False
    )


    query_features = extract_query_features(
        [query]
    )


    query_features_scaled = scaler.transform(
        query_features
    )


    # --------------------------------------------------------
    # Combine V2 features
    # --------------------------------------------------------

    combined_features = np.hstack([

        tfidf_features,

        embeddings,

        query_features_scaled
    ])


    # --------------------------------------------------------
    # V2 complexity prediction
    # --------------------------------------------------------

    complexity = model.predict(
        combined_features
    )[0]


    # --------------------------------------------------------
    # V2 confidence
    #
    # Confidence is recorded for analysis but does NOT
    # modify the V3 quality requirement.
    # --------------------------------------------------------

    probabilities = model.predict_proba(
        combined_features
    )[0]


    confidence = float(
        np.max(probabilities)
    )


    # --------------------------------------------------------
    # V3 routing
    # --------------------------------------------------------

    selected_model, routing_info = route_v3(
        complexity
    )


    # --------------------------------------------------------
    # Retrieve operational scores
    # --------------------------------------------------------

    small_score = routing_info["scores"].get(
        "small_model",
        np.nan
    )

    medium_score = routing_info["scores"].get(
        "medium_model",
        np.nan
    )

    large_score = routing_info["scores"].get(
        "large_model",
        np.nan
    )


    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append({

        "query": query,

        "complexity": complexity,

        "confidence": round(
            confidence,
            4
        ),

        "required_quality": round(
            routing_info["required_quality"],
            4
        ),

        "eligible_models":
            ", ".join(
                routing_info["eligible_models"]
            ),

        "selected_model":
            selected_model,

        "small_score":
            round(small_score, 4)
            if not np.isnan(small_score)
            else None,

        "medium_score":
            round(medium_score, 4)
            if not np.isnan(medium_score)
            else None,

        "large_score":
            round(large_score, 4)
            if not np.isnan(large_score)
            else None,

        "fallback":
            routing_info["fallback"]
    })


# ============================================================
# 6. RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)


print("\n")
print("=" * 120)
print("V3 ROUTING RESULTS")
print("=" * 120)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 7. MODEL SELECTION DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("MODEL SELECTION DISTRIBUTION")
print("=" * 80)

selection_distribution = (
    results_df["selected_model"]
    .value_counts()
)

print(
    selection_distribution
)


# ============================================================
# 8. COMPLEXITY DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("PREDICTED COMPLEXITY DISTRIBUTION")
print("=" * 80)

print(
    results_df["complexity"]
    .value_counts()
)


# ============================================================
# 9. DETAILED ROUTING DECISIONS
# ============================================================

print("\n")
print("=" * 120)
print("DETAILED ROUTING DECISIONS")
print("=" * 120)


for _, row in results_df.iterrows():

    complexity = row["complexity"]

    selected_model, routing_info = route_v3(
        complexity
    )


    print("\nQuery:")
    print(row["query"])

    print(
        f"Complexity: {complexity}"
    )

    print(
        f"Confidence: {row['confidence']}"
    )

    print(
        f"Required quality: "
        f"{routing_info['required_quality']:.2f}"
    )

    print(
        f"Eligible models: "
        f"{routing_info['eligible_models']}"
    )

    print(
        f"Selected model: "
        f"{selected_model}"
    )

    print("Operational scores:")

    for model_name, score in routing_info[
        "scores"
    ].items():

        print(
            f"  {model_name:15s} "
            f"→ {score:.4f}"
        )


# ============================================================
# 10. MODEL PROFILES TABLE
# ============================================================

print("\n")
print("=" * 90)
print("MODEL PROFILES")
print("=" * 90)


profile_df = pd.DataFrame(
    model_profiles
).T


print(
    profile_df[
        [
            "quality",
            "latency_ms",
            "cost",
            "normalized_cost",
            "normalized_latency"
        ]
    ].to_string()
)


# ============================================================
# 11. ROUTING POLICY SUMMARY
# ============================================================

print("\n")
print("=" * 90)
print("V3 ROUTING POLICY")
print("=" * 90)


for complexity, requirement in quality_requirement.items():

    eligible = [

        name

        for name, profile
        in model_profiles.items()

        if profile["quality"] >= requirement
    ]

    selected, _ = route_v3(
        complexity
    )

    print(
        f"{complexity:8s} | "
        f"min_quality={requirement:.2f} | "
        f"eligible={eligible} | "
        f"selected={selected}"
    )


# ============================================================
# 12. SAVE V3 ARTIFACTS
# ============================================================

save_dir = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing/"
    "notebooks/data/evaluation/v3"
)

os.makedirs(
    save_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# Save model profiles
# ------------------------------------------------------------

profiles_path = os.path.join(
    save_dir,
    "model_profiles.json"
)


with open(
    profiles_path,
    "w"
) as f:

    json.dump(
        model_profiles,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Save V3 configuration
# ------------------------------------------------------------

config = {

    "version": "V3",

    "description":
        "Quality-constrained cost/latency aware routing",

    "quality_requirement":
        quality_requirement,

    "cost_weight":
        COST_WEIGHT,

    "latency_weight":
        LATENCY_WEIGHT,

    "model_profiles":
        model_profiles
}


config_path = os.path.join(
    save_dir,
    "v3_config.json"
)


with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Save routing results
# ------------------------------------------------------------

results_path = os.path.join(
    save_dir,
    "v3_routing_test.csv"
)


results_df.to_csv(
    results_path,
    index=False
)


# ============================================================
# 13. FINAL STATUS
# ============================================================

print("\n")
print("=" * 90)
print("V3 COMPLETE")
print("=" * 90)

print(
    "\nArtifacts saved:"
)

print(
    f"1. Model profiles:\n   {profiles_path}"
)

print(
    f"2. V3 configuration:\n   {config_path}"
)

print(
    f"3. Routing results:\n   {results_path}"
)

print("\nV3 policy is now frozen.")

V3 — FINAL QUALITY / COST / LATENCY AWARE ROUTER


V3 ROUTING RESULTS
                                                                              query complexity  confidence  required_quality                        eligible_models selected_model  small_score  medium_score  large_score  fallback
                                                                    What is Python?     simple      0.5078              0.80 small_model, medium_model, large_model    small_model          0.0        0.2572          1.0     False
                                                     What is the capital of France?    complex      0.6696              0.95                            large_model    large_model          NaN           NaN          1.0     False
                                                        What is 25 multiplied by 4?    complex      0.4884              0.95                            large_model    large_model          NaN           NaN          1.0     False
              

In [6]:
import os

ROOT = "/home/sagemaker-user/adaptive-llm-routing"

def print_tree(path, prefix=""):
    items = sorted(os.listdir(path))

    for i, item in enumerate(items):
        full_path = os.path.join(path, item)
        is_last = i == len(items) - 1

        connector = "└── " if is_last else "├── "
        print(prefix + connector + item)

        if os.path.isdir(full_path):
            extension = "    " if is_last else "│   "
            print_tree(
                full_path,
                prefix + extension
            )

print(f"{ROOT}/")
print_tree(ROOT)

/home/sagemaker-user/adaptive-llm-routing/
├── .env.example
├── .github
│   └── workflows
│       └── ci.yml
├── .gitignore
├── Dockerfile
├── README.md
├── app
│   ├── agents
│   │   ├── __init__.py
│   │   ├── diagnosis.py
│   │   ├── issue_agent.py
│   │   ├── patch_generator.py
│   │   └── verification.py
│   ├── api
│   │   ├── __init__.py
│   │   └── routes.py
│   ├── cache
│   │   ├── __init__.py
│   │   └── redis_cache.py
│   ├── evaluation
│   │   ├── __init__.py
│   │   ├── __pycache__
│   │   │   ├── __init__.cpython-312.pyc
│   │   │   └── audit_labels.cpython-312.pyc
│   │   ├── audit_labels.py
│   │   ├── benchmark.py
│   │   ├── evaluator.py
│   │   └── metrics.py
│   ├── models
│   │   ├── __init__.py
│   │   ├── bedrock.py
│   │   └── registry.py
│   ├── observability
│   │   ├── __init__.py
│   │   ├── issue_agent.py
│   │   ├── logging.py
│   │   ├── metrics.py
│   │   └── tracing.py
│   └── router
│       ├── __init__.py
│       ├── __pycache__
│       │   ├── __ini